In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t090/task.json'
with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task data
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "product-catalog 流量下跌告警" (product-catalog traffic drop alert). It occurred between 2026-04-12 21:26:08 and 2026-04-12 21:36:32 (UTC+8). The entity that triggered the alert is the operation `product-catalog::oteldemo.ProductCatalogService/ListProducts` of type `apm.operation` in the `apm` domain. The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', {'start': '2026-04-12T21:26:08+08:00', 'end': '2026-04-12T21:36:32+08:00'}, {'entity_id': '455f921f0afc16175724d9f27faf18e4', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/ListProducts', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">product-catalog 流量下跌告警<alert_event event_id="62bdd34ccb277ea3a2a6c17384cf2c8c" trans_id="mc4bf4bdm5203588dneo5j9ou4" rule_id="e876641e-1e5e-49d4-9e38-9233255b5baf" rule_name="product-catalog 流量下跌告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t090/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their associated components
kpi_components = metrics_data[['metric', 'entity_name']].drop_duplicates()

kpi_components

```
Out[2]:
```
The `metrics.parquet` file contains 283 unique combinations of KPIs and their associated components. These include metrics such as `node_ready_status`, `node_memory_usage_rate`, `workload`, and `error_rate`, among others. The components range from nodes to services and operations, such as `shipping`, `frontend-proxy::ingress`, and specific pods like `frontend-proxy-67d64dc4f8-xz2fd`. To analyze anomalies, focus on relevant KPIs like `error_rate`, `workload`, and resource usage metrics.

The original code execution output of IPython Kernel is also provided below for reference:

metric                      entity_name
0           node_ready_status                                 
2175    node_not_ready_status                                 
3625   node_memory_usage_rate                                 
6380     node_disk_usage_rate                                 
7830   node_pod_running_count                                 
...                       ...                              ...
80882                workload                         shipping
91107              error_rate          frontend-proxy::ingress
91113              error_rate  frontend-proxy-67d64dc4f8-xz2fd
91115              error_rate  frontend-proxy-67d64dc4f8-8c99b
91117              error_rate  frontend-proxy-67d64dc4f8-4ptg6

[283 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
from pytz import timezone
from datetime import datetime

# Convert 'time' and 'value' columns to numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Calculate global thresholds (P95) for each 'component-KPI' time series
global_thresholds = metrics_data.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
global_thresholds.columns = ['entity_name', 'metric', 'P95']

# Define the time range for filtering (convert to UTC+8 timezone)
start_time = int(datetime.strptime('2026-04-12T21:26:08+08:00', '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)
end_time = int(datetime.strptime('2026-04-12T21:36:32+08:00', '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)

# Filter data within the specified time range
filtered_data = metrics_data[(metrics_data['time'] >= start_time) & (metrics_data['time'] <= end_time)]

# Focus on specific KPIs and possible root cause components
focused_kpis = ['error_rate', 'workload', 'node_memory_usage_rate', 'node_cpu_usage_rate', 'node_disk_usage_rate']
focused_data = filtered_data[filtered_data['metric'].isin(focused_kpis)]

# Display the global thresholds and the filtered data for analysis
global_thresholds, focused_data

```
Out[3]:
```
Global thresholds (P95) were calculated for each KPI-component combination across all data. The filtered data within the time range (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8) includes 1,197 records focusing on KPIs such as `error_rate`, `workload`, and resource usage metrics. These metrics are critical for identifying anomalies in components like `product-catalog`, `payment`, `quote`, and `shipping`. Further analysis should focus on deviations from the calculated P95 thresholds in this filtered dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(                                      entity_name                               metric          P95
0                                                     apiserver_concurrent_connections     6.000000
1                                                   apiserver_non_2xx_read_request_qps     0.161881
2                                                  apiserver_non_2xx_write_request_qps     0.000000
3                                                        apiserver_requests_per_second    21.150153
4                                                            cluster_failed_node_count     0.000000
..                                            ...                                  ...          ...
278   shipping::oteldemo.ShippingService/GetQuote          avg_request_latency_seconds     0.016888
279   shipping::oteldemo.ShippingService/GetQuote                        request_count  6333.600000
280   shipping::oteldemo.ShippingService/GetQuote                           slow_count    66.000000
281  shipping::oteldemo.ShippingService/ShipOrder          avg_request_latency_seconds     0.000010
282  shipping::oteldemo.ShippingService/ShipOrder                        request_count  6331.400000

[283 rows x 3 columns],                    time domain          entity_set entity_id      entity_name                  metric         value                                     metric_set_id service
3745   1776000368000000    k8s            k8s.node                             node_memory_usage_rate     29.883019  k8s@metric_set@k8s.metric.high_level_metric_node    None
3746   1776000398000000    k8s            k8s.node                             node_memory_usage_rate     29.838713  k8s@metric_set@k8s.metric.high_level_metric_node    None
3747   1776000428000000    k8s            k8s.node                             node_memory_usage_rate     29.794799  k8s@metric_set@k8s.metric.high_level_metric_node    None
3748   1776000458000000    k8s            k8s.node                             node_memory_usage_rate     29.864478  k8s@metric_set@k8s.metric.high_level_metric_node    None
3749   1776000488000000    k8s            k8s.node                             node_memory_usage_rate     29.801800  k8s@metric_set@k8s.metric.high_level_metric_node    None
...                 ...    ...                 ...       ...              ...                     ...           ...                                               ...     ...
90804  1776000968000000    apm  apm.service.legacy                    payment                workload   5097.000000                                              None    None
90808  1776000968000000    apm  apm.service.legacy            product-catalog                workload  17522.000000                                              None    None
90812  1776000968000000    apm  apm.service.legacy                      quote                workload   5093.000000                                              None    None
90817  1776000968000000    apm  apm.service.legacy             recommendation                workload    224.000000                                              None    None
90822  1776000968000000    apm  apm.service.legacy                   shipping                workload  10191.000000                                              None    None

[1197 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered dataset with the global thresholds to compare KPI values against P95 thresholds
merged_data = pd.merge(focused_data, global_thresholds, on=['entity_name', 'metric'], how='left')

# Identify anomalies where the KPI value exceeds the P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['P95']]

# Focus on specific components and KPIs
target_components = ['product-catalog', 'payment', 'quote', 'shipping']
target_anomalies = anomalies[anomalies['entity_name'].str.contains('|'.join(target_components))]

# Display the anomalies for analysis
target_anomalies

```
Out[4]:
```
Anomalies were identified in the filtered dataset where KPI values exceeded their P95 thresholds. Specifically:

1. The `payment` service's `workload` exceeded its P95 threshold of 6331.4 twice, with values of 6339.0 and 6617.0.
2. The `quote` service's `workload` exceeded its P95 threshold of 6333.6, with a value of 6615.0.
3. The `shipping` service's `workload` exceeded its P95 threshold of 12666.8, with a value of 13231.0.

These anomalies indicate potential issues in these services during the analyzed time window.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id entity_name    metric    value metric_set_id service      P95
892  1776000368000000    apm  apm.service.legacy               payment  workload   6339.0          None    None   6331.4
982  1776000548000000    apm  apm.service.legacy               payment  workload   6617.0          None    None   6331.4
984  1776000548000000    apm  apm.service.legacy                 quote  workload   6615.0          None    None   6333.6
986  1776000548000000    apm  apm.service.legacy              shipping  workload  13231.0          None    None  12666.8```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t090/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Convert 'startTime' and 'endTime' to numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Define the time range for filtering (convert to nanoseconds for traces)
start_time_ns = start_time * 1_000
end_time_ns = end_time * 1_000

# Filter traces within the specified time range
filtered_traces = traces_data[
    (traces_data['startTime'] >= start_time_ns) & (traces_data['endTime'] <= end_time_ns)
]

# Focus on traces involving the target services
target_services = ['product-catalog', 'payment', 'quote', 'shipping']
focused_traces = filtered_traces[filtered_traces['serviceName'].str.contains('|'.join(target_services), na=False)]

# Identify the most downstream faulty service in the call chain
# A downstream service is typically identified by the absence of a parentSpanId in the trace
downstream_faulty_traces = focused_traces[focused_traces['parentSpanId'].isna()]

downstream_faulty_traces

```
Out[5]:
```
No downstream faulty traces were identified within the specified time range (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8) for the `product-catalog`, `payment`, `quote`, and `shipping` services. This indicates that there were no traces without a `parentSpanId` involving these services during the analyzed period. Further investigation may be needed to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, events, __topic__, __tag__:__pack_id__, ip, links]
Index: []

[0 rows x 21 columns]```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t090/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert '_time_' column to numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs within the specified time range
filtered_logs = logs_data[(logs_data['_time_'] >= start_time) & (logs_data['_time_'] <= end_time)]

# Focus on logs related to the target services
focused_logs = filtered_logs[
    filtered_logs['_container_name_'].str.contains('|'.join(target_components), na=False)
]

# Search for error messages or anomalies in the logs
error_logs = focused_logs[focused_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[6]:
```
No error logs or anomalies were found within the specified time range (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8) for the `product-catalog`, `payment`, `quote`, and `shipping` services. This suggests that no explicit error messages were logged during the analyzed period for these services. Further investigation may be required to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Load the events.parquet file
events_file_path = 'rca100/cases/t090/events.parquet'
events_data = pd.read_parquet(events_file_path)

# Convert event time to numeric for filtering
events_data['event_time'] = pd.to_numeric(events_data['eventId'].apply(lambda x: json.loads(x).get('metadata', {}).get('creationTimestamp', 0)), errors='coerce')

# Filter events within the specified time range
filtered_events = events_data[(events_data['event_time'] >= start_time) & (events_data['event_time'] <= end_time)]

# Focus on events related to the target services
focused_events = filtered_events[
    filtered_events['pod_name'].str.contains('|'.join(target_components), na=False)
]

# Look for resource or scheduling issues in the events
resource_issues = focused_events[
    focused_events['eventId'].str.contains('Failed|OOM|Unschedulable|CrashLoopBackOff', case=False, na=False)
]

resource_issues

```
Out[7]:
```
No resource or scheduling issues were found in the Kubernetes events within the specified time range (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8) for the `product-catalog`, `payment`, `quote`, and `shipping` services. This indicates that there were no reported problems such as failed scheduling, out-of-memory (OOM) errors, or crash loops during the analyzed period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [eventId, hostname, level, pod_id, pod_name, clusterId, clusterName, __topic__, event_time]
Index: []```
```